In [14]:
import re
import os
import email
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [15]:
BASE_DIR   = Path(os.getcwd()).parent  # src se bahar nikalta hai
RAW_CSV    = BASE_DIR / "data" / "raw" / "emails.csv"
OUTPUT_CSV = BASE_DIR / "data" / "processed" / "emails_clean.csv"

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print("Base Dir:", BASE_DIR)
print("Raw CSV :", RAW_CSV)
print("Output  :", OUTPUT_CSV)

Base Dir: d:\deep_learning_projects\Smart_email_reply\smart_email_reply
Raw CSV : d:\deep_learning_projects\Smart_email_reply\smart_email_reply\data\raw\emails.csv
Output  : d:\deep_learning_projects\Smart_email_reply\smart_email_reply\data\processed\emails_clean.csv


In [16]:
print("Loading emails.csv ...")
df = pd.read_csv(RAW_CSV, nrows=30000)
print(f"Loaded {len(df):,} rows")
df.head(3)

Loading emails.csv ...
Loaded 30,000 rows


,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...


In [17]:
def parse_raw_message(raw):
    try:
        msg     = email.message_from_string(str(raw))
        subject = msg.get("Subject", "").strip()
        sender  = msg.get("From",    "").strip()
        to      = msg.get("To",      "").strip()
        date    = msg.get("Date",    "").strip()
        body    = msg.get_payload()
        if isinstance(body, list):
            body = " ".join(str(p.get_payload()) for p in body)
        return {"sender": sender, "to": to, "subject": subject, "date": date, "body": str(body)}
    except:
        return {"sender":"","to":"","subject":"","date":"","body":""}

print(" Parsing messages ...")
parsed = df["message"].apply(parse_raw_message)
df_parsed = pd.DataFrame(list(parsed))
print(f"Parsed {len(df_parsed):,} emails")
df_parsed.head(3)

 Parsing messages ...
Parsed 30,000 emails


,sender,to,subject,date,body
0,phillip.allen@enron.com,tim.belden@enron.com,,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",Here is our forecast\n\n
1,phillip.allen@enron.com,john.lavorato@enron.com,Re:,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",Traveling to have a business meeting takes the...
2,phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",test successful. way to go!!!


In [18]:
def clean_body(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"-{2,}.*?(Original Message|Forwarded).*?-{2,}", "", text, flags=re.I|re.S)
    text = re.sub(r"^\s*>.*$", "", text, flags=re.M)
    text = re.sub(r"\S+@\S+\.\S+", "", text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

print(" Cleaning body text ...")
df_parsed["body_clean"] = df_parsed["body"].apply(clean_body)
print("Done!")
df_parsed[["subject","body_clean"]].head(3)

 Cleaning body text ...
Done!


,subject,body_clean
0,,Here is our forecast
1,Re:,Traveling to have a business meeting takes the...
2,Re: test,test successful. way to go!!!


In [19]:
#filtering and removing duplicates
before = len(df_parsed)

df_parsed = df_parsed[df_parsed["body_clean"].str.len() >= 50]
df_parsed = df_parsed[df_parsed["body_clean"].str.len() <= 1000]
df_parsed.drop_duplicates(subset=["body_clean"], inplace=True)
df_parsed.reset_index(drop=True, inplace=True)
print(f"Before filtering : {before:,}")
print(f"After filtering  : {len(df_parsed):,}")
print(f"Removed          : {before - len(df_parsed):,} rows")

Before filtering : 30,000
After filtering  : 8,064
Removed          : 21,936 rows


In [20]:
cols = ["sender","to","subject","date","body_clean"]
df_parsed[cols].to_csv(OUTPUT_CSV, index=False)
print(f"Saved → {OUTPUT_CSV}")
print(f"Total clean emails: {len(df_parsed):,}")

Saved → d:\deep_learning_projects\Smart_email_reply\smart_email_reply\data\processed\emails_clean.csv
Total clean emails: 8,064
